# Jeff Snake on the ANE

Serve the base build with the Snake build as an adapter and let Jeff play a few self-play games from fixed openings, one decision per move.

In [ ]:
import os, sys, time, statistics
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "jeff_nb.py").exists() else Path.cwd() / "notebooks"))
import jeff_nb as J
print("repo:", J.REPO)
print("models dir:", J.models_dir())
print("COREAI_PYTHON:", os.environ.get("COREAI_PYTHON", "<not set>"))

In [ ]:
base, snake_build = J.ensure("base"), J.ensure("snake")
if not J.reuse_url():
    for b in (base, snake_build):
        J.compile_build(b)
proc, url = J.start_server(base, {"snake": snake_build})   # the server's --model is the base build
print(J.health(url)["backend"], J.health(url)["adapters"])

In [ ]:
import random
from jeff_snake import INSTRUCTIONS, OPTIONS, opening_states, render_state, board_text, play_game, summarize_games

adapter = "snake"
timings = []
def choose(snake, food):
    body = {"model": "jeff-latest", "state": render_state(snake, food),
            "questions": {"move": {"type": "choice", "instructions": INSTRUCTIONS, "criteria": dict(OPTIONS)}}}
    if adapter:
        body["adapter"] = adapter
    out = J.decide(url, body)
    timings.append(out["timings"]["total_ms"])
    return out["answers"]["move"]["choice"]

games = [play_game(choose, s, f, random.Random(i)) for i, (s, f) in enumerate(opening_states(4, seed=0))]
print(summarize_games(games))
print(f"{len(timings)} decisions, median {statistics.median(timings):.1f} ms")

In [ ]:
J.stop_server(proc)

## Next steps

Training for new tasks happens off the ANE (PyTorch on a Mac with MPS, or a GPU such as Colab). The resulting adapter and choice head are then merged, converted to Core AI, compiled and served on the Apple Neural Engine on the Mac (see the README, "Build your own"). Colab and other non-Apple machines cannot compile or serve on the ANE.

## Credits

- **Base model:** Jeff v1.3, a fine-tune of Qwen3.5-0.8B by its author, [mstrasser/jeff-base](https://huggingface.co/mstrasser/jeff-base) (Apache-2.0).
- **Backbone:** Qwen3.5-0.8B by the Qwen Team, Alibaba Cloud ([Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B), Apache-2.0).
- **Snake adapter:** a rank-16 LoRA plus choice head trained for the base model on 2026-10-07 and merged before conversion; the build shipped here comes from a local Apple M5 Max run (PyTorch MPS, `forge.py jeff-train-lora --task snake`). A separate Unsloth Snake training on a Colab A100 (also 2026-10-07) exists as an experiment; it is not what this notebook serves.
- **Tetris adapter:** LoRA plus choice head trained locally on an Apple M5 Max with PyTorch MPS, using this repo's hand-rolled LoRA (no Unsloth), in the bounded overnight rounds of 2026-10-08 (round 02 shipped). A Colab A100 continuation on 2026-10-07 failed and was discarded.
- **Training data:** generated by this repo (El-Tetris oracle for Tetris, a search oracle for Snake); no third-party dataset.